In [1]:
# =====================================================
# CELL 1 — SETUP + PREPROCESSING
# =====================================================

import re
import html
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ===============================
# PREPROCESS
# ===============================

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = html.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()

    text = "[CLS] " + text + " [SEP]"
    return text


# ===============================
# LOAD DATASET
# ===============================

dataset_a = pd.read_csv("data_80.csv")
dataset_b = pd.read_csv("data_20.csv")

X_a = dataset_a["Text"]
y_a = dataset_a["label"]

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_a,
    y_a,
    test_size=0.125,
    random_state=42,
    stratify=y_a
)

X_test_text = dataset_b["Text"]
y_test = dataset_b["label"]

print("Preprocessing...")

X_train_text = [preprocess_text(t) for t in tqdm(X_train_text)]
X_val_text   = [preprocess_text(t) for t in tqdm(X_val_text)]
X_test_text  = [preprocess_text(t) for t in tqdm(X_test_text)]


# ===============================
# TOKENIZER
# ===============================

model_name = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_name)


# ===============================
# DATASET CLASS
# ===============================

class IndoBERTDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):

        self.texts = texts
        self.labels = labels.values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

print("Setup selesai.")

c:\SKRIPSI BRYAN\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Preprocessing...


100%|██████████| 3523/3523 [00:00<00:00, 116554.39it/s]


Setup selesai.


In [2]:
# =====================================================
# CELL 2 — HYPERPARAMETER TUNING (F1 VERSION)
# =====================================================

from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score
import json

MAX_HPT_SAMPLES = 5000
RANDOM_STATE = 42

dataset_hpt = dataset_a.sample(
    n=min(MAX_HPT_SAMPLES, len(dataset_a)),
    random_state=RANDOM_STATE
)

X_hpt = [preprocess_text(t) for t in dataset_hpt["Text"]]
y_hpt = dataset_hpt["label"]

X_train_hpt, X_val_hpt, y_train_hpt, y_val_hpt = train_test_split(
    X_hpt,
    y_hpt,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_hpt
)

param_grid = {
    "lr": [2e-5, 3e-5, 5e-5],
    "batch_size": [16, 32],
    "max_len": [128],
    "epochs": [10]
}

results = []

print("🚀 Hyperparameter tuning (F1) dimulai...")

for params in ParameterGrid(param_grid):

    print("\nTesting:", params)

    train_loader = DataLoader(
        IndoBERTDataset(X_train_hpt, y_train_hpt, tokenizer, params["max_len"]),
        batch_size=params["batch_size"],
        shuffle=True
    )

    val_loader = DataLoader(
        IndoBERTDataset(X_val_hpt, y_val_hpt, tokenizer, params["max_len"]),
        batch_size=params["batch_size"]
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=1
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"])
    criterion = nn.BCEWithLogitsLoss()

    # ===== TRAIN =====
    for epoch in range(params["epochs"]):

        model.train()

        for batch in train_loader:

            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=mask)

            logits = outputs.logits.squeeze()
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

    # ===== VALIDATION =====
    model.eval()

    preds = []
    labels_list = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=mask)

            probs = torch.sigmoid(outputs.logits.squeeze())
            pred = (probs >= 0.5).cpu().numpy()

            preds.extend(pred)
            labels_list.extend(batch["labels"].numpy())

    # ===== HITUNG F1 =====
    f1 = f1_score(labels_list, preds)

    print("F1:", f1)

    results.append({
        "params": params,
        "f1": f1
    })

# ===== AMBIL YANG TERBAIK =====
best = max(results, key=lambda x: x["f1"])

print("\n✅ BEST PARAMS:", best)

with open("hyperparameters_indobert_f1.json", "w") as f:
    json.dump(best, f, indent=4)

print("💾 Saved to hyperparameters_indobert_f1.json")

🚀 Hyperparameter tuning (F1) dimulai...

Testing: {'batch_size': 16, 'epochs': 10, 'lr': 2e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1040.85it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


F1: 0.7810026385224275

Testing: {'batch_size': 16, 'epochs': 10, 'lr': 3e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 992.36it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


F1: 0.749379652605459

Testing: {'batch_size': 16, 'epochs': 10, 'lr': 5e-05, 'max_len': 128}


...

F1: 0.7586206896551724

Testing: {'batch_size': 32, 'epochs': 10, 'lr': 3e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1015.44it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


F1: 0.7192982456140351

Testing: {'batch_size': 32, 'epochs': 10, 'lr': 5e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 963.20it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


F1: 0.7580645161290323

✅ BEST PARAMS: {'params': {'batch_size': 16, 'epochs': 10, 'lr': 2e-05, 'max_len': 128}, 'f1': 0.7810026385224275}
💾 Saved to hyperparameters_indobert_f1.json


**recall**

In [3]:
# =====================================================
# CELL 3 — FINAL TRAINING
# =====================================================

# ===============================
# LOAD HPT PARAMS IF EXIST
# ===============================

try:

    with open("hyperparameters_indobert_f1.json") as f:
        best = json.load(f)["params"]

    lr = best["lr"]
    batch_size = best["batch_size"]
    max_len = best["max_len"]
    epochs = best["epochs"]

    print("Using tuned hyperparameters:", best)

except:

    print("No HPT file found. Using default params.")

    lr = 2e-5
    batch_size = 16
    max_len = 128
    epochs = 5


# ===============================
# DATALOADER
# ===============================

train_loader = DataLoader(
    IndoBERTDataset(X_train_text, y_train, tokenizer, max_len),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    IndoBERTDataset(X_val_text, y_val, tokenizer, max_len),
    batch_size=batch_size
)

test_loader = DataLoader(
    IndoBERTDataset(X_test_text, y_test, tokenizer, max_len),
    batch_size=batch_size
)


# ===============================
# MODEL
# ===============================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.BCEWithLogitsLoss()

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    0,
    total_steps
)

patience = 2
best_val_recall = 0
counter = 0
best_model_state = None

print("🚀 Final training dimulai...")


for epoch in range(epochs):

    model.train()

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        logits = outputs.logits.squeeze()
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()


    # validation
    model.eval()

    preds = []
    labels_list = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=mask)

            probs = torch.sigmoid(outputs.logits.squeeze())
            pred = (probs >= 0.5).cpu().numpy()

            preds.extend(pred)
            labels_list.extend(labels.cpu().numpy())

    val_recall = recall_score(labels_list, preds)

    print(f"\nEpoch {epoch+1} Recall: {val_recall:.4f}")

    if val_recall > best_val_recall:

        best_val_recall = val_recall
        best_model_state = model.state_dict()
        counter = 0
        print("✅ Model terbaik diperbarui")

    else:

        counter += 1
        if counter >= patience:
            print("🛑 Early stopping")
            break


model.load_state_dict(best_model_state)
model.eval()


# ===============================
# FINAL TEST
# ===============================

all_preds = []
all_probs = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        probs = torch.sigmoid(outputs.logits.squeeze())

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend((probs >= 0.5).cpu().numpy())


y_pred = np.array(all_preds)
y_proba = np.array(all_probs)

print("\n=== FINAL TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_proba))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

Using tuned hyperparameters: {'batch_size': 16, 'epochs': 10, 'lr': 2e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1002.34it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Final training dimulai...


100%|██████████| 771/771 [02:39<00:00,  4.84it/s]



Epoch 1 Recall: 0.9107
✅ Model terbaik diperbarui


...


Epoch 2 Recall: 0.8185


100%|██████████| 771/771 [02:40<00:00,  4.82it/s]



Epoch 3 Recall: 0.7381
🛑 Early stopping


100%|██████████| 221/221 [00:13<00:00, 16.18it/s]


=== FINAL TEST ===
Accuracy : 0.930173147885325
Precision: 0.8803571428571428
Recall   : 0.7336309523809523
F1-score : 0.8003246753246753
AUC      : 0.9515891458302015

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0     0.9396    0.9765    0.9577      2851
           1     0.8804    0.7336    0.8003       672

    accuracy                         0.9302      3523
   macro avg     0.9100    0.8551    0.8790      3523
weighted avg     0.9283    0.9302    0.9277      3523

=== CONFUSION MATRIX ===
[[2784   67]
 [ 179  493]]


**loss**

In [2]:
# =====================================================
# CELL 3 — FINAL TRAINING (EARLY STOPPING = VAL LOSS)
# =====================================================

# ===============================
# LOAD HPT PARAMS IF EXIST
# ===============================

try:

    with open("hyperparameters_indobert.json") as f:
        best = json.load(f)["params"]

    lr = best["lr"]
    batch_size = best["batch_size"]
    max_len = best["max_len"]
    epochs = best["epochs"]

    print("Using tuned hyperparameters:", best)

except:

    print("No HPT file found. Using default params.")

    lr = 2e-5
    batch_size = 16
    max_len = 128
    epochs = 5


# ===============================
# DATALOADER
# ===============================

train_loader = DataLoader(
    IndoBERTDataset(X_train_text, y_train, tokenizer, max_len),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    IndoBERTDataset(X_val_text, y_val, tokenizer, max_len),
    batch_size=batch_size
)

test_loader = DataLoader(
    IndoBERTDataset(X_test_text, y_test, tokenizer, max_len),
    batch_size=batch_size
)


# ===============================
# MODEL
# ===============================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.BCEWithLogitsLoss()

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    0,
    total_steps
)

patience = 2
best_val_loss = float("inf")
counter = 0
best_model_state = None

print("🚀 Final training dimulai...")


for epoch in range(epochs):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        logits = outputs.logits.squeeze()
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)


    # ================= VALIDATION =================
    model.eval()

    val_loss = 0
    preds = []
    labels_list = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=mask)

            logits = outputs.logits.squeeze()

            loss = criterion(logits, labels)
            val_loss += loss.item()

            probs = torch.sigmoid(logits)
            pred = (probs >= 0.5).cpu().numpy()

            preds.extend(pred)
            labels_list.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")


    # ================= EARLY STOPPING =================

    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_model_state = model.state_dict()
        counter = 0

        print("✅ Model terbaik diperbarui")

    else:

        counter += 1
        print(f"⚠️ Tidak ada peningkatan ({counter}/{patience})")

        if counter >= patience:
            print("🛑 Early stopping")
            break


model.load_state_dict(best_model_state)
model.eval()


# ===============================
# FINAL TEST
# ===============================

all_preds = []
all_probs = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        probs = torch.sigmoid(outputs.logits.squeeze())

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend((probs >= 0.5).cpu().numpy())


y_pred = np.array(all_preds)
y_proba = np.array(all_probs)

print("\n=== FINAL TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_proba))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

Using tuned hyperparameters: {'batch_size': 16, 'epochs': 5, 'lr': 5e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1011.01it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Final training dimulai...


100%|██████████| 771/771 [02:42<00:00,  4.75it/s]



Epoch 1
Train Loss: 0.2636
Validation Loss: 0.2440
✅ Model terbaik diperbarui


...


Epoch 3
Train Loss: 0.1210
Validation Loss: 0.2031
⚠️ Tidak ada peningkatan (1/2)


100%|██████████| 771/771 [02:43<00:00,  4.71it/s]



Epoch 4
Train Loss: 0.0743
Validation Loss: 0.2200
⚠️ Tidak ada peningkatan (2/2)
🛑 Early stopping


100%|██████████| 221/221 [00:13<00:00, 16.22it/s]



=== FINAL TEST ===
Accuracy : 0.9256315640079478
Precision: 0.8660714285714286
Recall   : 0.7217261904761905
F1-score : 0.7873376623376623
AUC      : 0.9279523893036695

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0     0.9369    0.9737    0.9549      2851
           1     0.8661    0.7217    0.7873       672

    accuracy                         0.9256      3523
   macro avg     0.9015    0.8477    0.8711      3523
weighted avg     0.9234    0.9256    0.9230      3523

=== CONFUSION MATRIX ===
[[2776   75]
 [ 187  485]]


**f1**

In [3]:
# =====================================================
# CELL 3 — FINAL TRAINING (EARLY STOPPING = VAL F1)
# =====================================================

# ===============================
# LOAD HPT PARAMS IF EXIST
# ===============================

try:

    with open("hyperparameters_indobert.json") as f:
        best = json.load(f)["params"]

    lr = best["lr"]
    batch_size = best["batch_size"]
    max_len = best["max_len"]
    epochs = best["epochs"]

    print("Using tuned hyperparameters:", best)

except:

    print("No HPT file found. Using default params.")

    lr = 2e-5
    batch_size = 16
    max_len = 128
    epochs = 5


# ===============================
# DATALOADER
# ===============================

train_loader = DataLoader(
    IndoBERTDataset(X_train_text, y_train, tokenizer, max_len),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    IndoBERTDataset(X_val_text, y_val, tokenizer, max_len),
    batch_size=batch_size
)

test_loader = DataLoader(
    IndoBERTDataset(X_test_text, y_test, tokenizer, max_len),
    batch_size=batch_size
)


# ===============================
# MODEL
# ===============================

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=1
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.BCEWithLogitsLoss()

total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    0,
    total_steps
)

patience = 2
best_val_f1 = 0
counter = 0
best_model_state = None

print("🚀 Final training dimulai...")


for epoch in range(epochs):

    # ================= TRAIN =================
    model.train()
    train_loss = 0

    for batch in tqdm(train_loader):

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        logits = outputs.logits.squeeze()
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)


    # ================= VALIDATION =================
    model.eval()

    preds = []
    labels_list = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=mask)

            probs = torch.sigmoid(outputs.logits.squeeze())
            pred = (probs >= 0.5).cpu().numpy()

            preds.extend(pred)
            labels_list.extend(labels.cpu().numpy())

    val_f1 = f1_score(labels_list, preds)

    print(f"\nEpoch {epoch+1}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation F1: {val_f1:.4f}")


    # ================= EARLY STOPPING =================

    if val_f1 > best_val_f1:

        best_val_f1 = val_f1
        best_model_state = model.state_dict()
        counter = 0

        print("✅ Model terbaik diperbarui")

    else:

        counter += 1
        print(f"⚠️ Tidak ada peningkatan ({counter}/{patience})")

        if counter >= patience:
            print("🛑 Early stopping")
            break


model.load_state_dict(best_model_state)
model.eval()


# ===============================
# FINAL TEST
# ===============================

all_preds = []
all_probs = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=mask)

        probs = torch.sigmoid(outputs.logits.squeeze())

        all_probs.extend(probs.cpu().numpy())
        all_preds.extend((probs >= 0.5).cpu().numpy())


y_pred = np.array(all_preds)
y_proba = np.array(all_probs)

print("\n=== FINAL TEST ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))
print("AUC      :", roc_auc_score(y_test, y_proba))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred, digits=4))

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

Using tuned hyperparameters: {'batch_size': 16, 'epochs': 5, 'lr': 5e-05, 'max_len': 128}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1022.91it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Final training dimulai...


100%|██████████| 771/771 [02:44<00:00,  4.70it/s]



Epoch 1
Train Loss: 0.2322
Validation F1: 0.8087
✅ Model terbaik diperbarui


...


Epoch 4
Train Loss: 0.0511
Validation F1: 0.8264
✅ Model terbaik diperbarui


100%|██████████| 771/771 [02:44<00:00,  4.69it/s]



Epoch 5
Train Loss: 0.0229
Validation F1: 0.8173
⚠️ Tidak ada peningkatan (1/2)


100%|██████████| 221/221 [00:13<00:00, 16.02it/s]


=== FINAL TEST ===
Accuracy : 0.9284700539313085
Precision: 0.8365384615384616
Recall   : 0.7767857142857143
F1-score : 0.8055555555555556
AUC      : 0.9422988592139766

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0     0.9483    0.9642    0.9562      2851
           1     0.8365    0.7768    0.8056       672

    accuracy                         0.9285      3523
   macro avg     0.8924    0.8705    0.8809      3523
weighted avg     0.9269    0.9285    0.9274      3523

=== CONFUSION MATRIX ===
[[2749  102]
 [ 150  522]]
